# Abaqus 封头加筋模型
- 本文档介绍了如何在 Abaqus 中创建一个封头加筋模型，进行有限元特征值屈曲仿真。
- 封头加筋是一种常见的结构增强方法，通常用于提高封头的强度和刚度。
- 以下代码需要依赖abaqus模块，请确保你已经安装了Abaqus软件并正确配置了Python环境。

In [ ]:
from abaqus import *
from abaqusConstants import *
from caeModules import *
import math
import regionToolset
from driverUtils import executeOnCaeStartup
executeOnCaeStartup()
Mdb()

model_name = 'Model-1'
model = mdb.models[model_name]


## 1) 全局参数定义
- 统一管理并修改输入参数，后续若需循环运行，可以改成数组或其他形式的输入。

In [ ]:
# 几何参数（单位：mm）
D = 2000.0              # 封头直径
H = D / 4.0             # 标准 2:1 椭圆封头冠深
a = D / 2.0             # 椭圆径向半轴
b = H                   # 椭圆轴向半轴

# 纵向加筋参数
n_long_ribs = 16
long_offset = 20.0
long_theta_start_deg = 20.0
long_theta_end_deg = 90.0
long_revolve_angle_deg = 3.0 

# 环向加筋参数
n_ring_ribs = 8
ring_span_deg = 4.0           
ring_offset = 20.0


n_head_pts = 60
n_rib_pts = 20


## 2) 以函数形式进行草图绘制
- 通过数学方法定义封头的几何形状和加筋位置，便于后续几何创建和装配。
- 函数形式可以方便调取，但在abaqus命令行中因缩进问题无法直接运行。

In [ ]:
# 按数量在纵向起止角之间均布环向筋位置
if n_ring_ribs <= 0:
    ring_theta_deg = []
elif n_ring_ribs == 1:
    ring_theta_deg = [0.5 * (long_theta_start_deg + long_theta_end_deg)]
else:
    step = (long_theta_end_deg - long_theta_start_deg) / float(n_ring_ribs - 1)
    ring_theta_deg = [long_theta_start_deg + i * step for i in range(n_ring_ribs)]


def ellipse_point_and_normal(theta):
    r0 = a * math.sin(theta)
    y0 = b * math.cos(theta)
    nx = r0 / (a * a)
    ny = y0 / (b * b)
    nlen = math.sqrt(nx * nx + ny * ny)
    nx = nx / nlen
    ny = ny / nlen
    return r0, y0, nx, ny

def unique_sorted(vals, eps=1e-9):
    vals = sorted(vals)
    out = []
    for v in vals:
        if (not out) or abs(v - out[-1]) > eps:
            out.append(v)
    return out

def ellipse_point(theta, offset):
    r0, y0, nx, ny = ellipse_point_and_normal(theta)
    return (r0 + offset * nx, y0 + offset * ny)

def ellipse_segment(theta1, theta2, offset, npts):
    pts = []
    for i in range(npts + 1):
        t = theta1 + (theta2 - theta1) * float(i) / float(npts)
        pts.append(ellipse_point(t, offset))
    return tuple(pts)


## 3) 旋转生成封头壳体
- 通过 360° 回转生成轴对称封头壳体。

In [ ]:
s1 = model.ConstrainedSketch(name='head_shell_sketch', sheetSize=4.0 * D)
s1.setPrimaryObject(option=STANDALONE)
s1.ConstructionLine(point1=(0.0, -2.0 * D), point2=(0.0, 2.0 * D))

base_thetas = [0.5 * math.pi * float(i) / float(n_head_pts) for i in range(n_head_pts + 1)]
head_pts = tuple(ellipse_point(t, 0.0) for t in base_thetas)
s1.Spline(points=head_pts)

p1 = model.Part(name='HeadShell', dimensionality=THREE_D, type=DEFORMABLE_BODY)
p1.BaseShellRevolve(sketch=s1, angle=360.0, flipRevolveDirection=OFF)
s1.unsetPrimaryObject()
del model.sketches['head_shell_sketch']


## 4) 生成环向筋（CircRingRibs）
- 将多条短线 360° 回转，得到环向筋壳面。
- 与壳体分部件创建，以便分截面控制厚度

In [ ]:
p_ring = None
if n_ring_ribs > 0:
    s_ring = model.ConstrainedSketch(name='circ_ring_rib_sketch', sheetSize=4.0 * D)
    s_ring.setPrimaryObject(option=STANDALONE)
    s_ring.ConstructionLine(point1=(0.0, -2.0 * D), point2=(0.0, 2.0 * D))

    for tc_deg in ring_theta_deg:
        tc = math.radians(tc_deg)
        r0, y0, nx, ny = ellipse_point_and_normal(tc)
        p_base = (r0, y0)
        p_tip = (r0 + ring_offset * nx, y0 + ring_offset * ny)
        s_ring.Line(point1=p_base, point2=p_tip)

    p_ring = model.Part(name='CircRingRibs', dimensionality=THREE_D, type=DEFORMABLE_BODY)
    p_ring.BaseShellRevolve(sketch=s_ring, angle=360.0, flipRevolveDirection=OFF)
    s_ring.unsetPrimaryObject()
    del model.sketches['circ_ring_rib_sketch']


## 5) 生成单条纵向筋（LongitudinalRib）
- 用“母线 + 偏置线 + 两端连接线”形成闭合截面。
- 直接生成平面壳面，后续在装配里做环形阵列。

In [ ]:
s2 = model.ConstrainedSketch(name='longitudinal_rib_sketch', sheetSize=4.0 * D)
s2.setPrimaryObject(option=STANDALONE)
s2.ConstructionLine(point1=(0.0, -2.0 * D), point2=(0.0, 2.0 * D))

t_start = math.radians(long_theta_start_deg)
t_end = math.radians(long_theta_end_deg)
base_pts = ellipse_segment(t_start, t_end, 0.0, n_head_pts)
tip_pts = ellipse_segment(t_start, t_end, long_offset, n_head_pts)

s2.Spline(points=base_pts)
s2.Spline(points=tuple(reversed(tip_pts)))
s2.Line(point1=base_pts[0], point2=tip_pts[0])
s2.Line(point1=base_pts[-1], point2=tip_pts[-1])

p2 = model.Part(name='LongitudinalRib', dimensionality=THREE_D, type=DEFORMABLE_BODY)
p2.BaseShell(sketch=s2)
s2.unsetPrimaryObject()
del model.sketches['longitudinal_rib_sketch']


## 6) 材料、截面、截面赋予
- 定义材料弹性参数（E、ν）。
- 分别给封头、环向筋、纵向筋赋壳截面厚度。

In [ ]:
mat_name = 'Steel'
head_sec_name = 'ShellSection-Head'
ring_sec_name = 'ShellSection-RingRib'
long_sec_name = 'ShellSection-LongRib'

head_thickness_mm = 10.0
ring_thickness_mm = 10.0
long_thickness_mm = 10.0

if mat_name not in model.materials:
    mat = model.Material(name=mat_name)
    mat.Elastic(table=((200000.0, 0.3),))

for sec_name in (head_sec_name, ring_sec_name, long_sec_name):
    if sec_name in model.sections:
        del model.sections[sec_name]

model.HomogeneousShellSection(
    name=head_sec_name,
    material=mat_name,
    thicknessType=UNIFORM,
    thickness=head_thickness_mm,
    idealization=NO_IDEALIZATION,
    poissonDefinition=DEFAULT,
    thicknessModulus=None,
    temperature=GRADIENT,
    useDensity=OFF,
    integrationRule=SIMPSON,
    numIntPts=5,
)

model.HomogeneousShellSection(
    name=ring_sec_name,
    material=mat_name,
    thicknessType=UNIFORM,
    thickness=ring_thickness_mm,
    idealization=NO_IDEALIZATION,
    poissonDefinition=DEFAULT,
    thicknessModulus=None,
    temperature=GRADIENT,
    useDensity=OFF,
    integrationRule=SIMPSON,
    numIntPts=5,
)

model.HomogeneousShellSection(
    name=long_sec_name,
    material=mat_name,
    thicknessType=UNIFORM,
    thickness=long_thickness_mm,
    idealization=NO_IDEALIZATION,
    poissonDefinition=DEFAULT,
    thicknessModulus=None,
    temperature=GRADIENT,
    useDensity=OFF,
    integrationRule=SIMPSON,
    numIntPts=5,
)

region_p1 = regionToolset.Region(faces=p1.faces)
p1.SectionAssignment(
    region=region_p1,
    sectionName=head_sec_name,
    offset=0.0,
    offsetType=MIDDLE_SURFACE,
    offsetField='',
    thicknessAssignment=FROM_SECTION,
)

if p_ring is not None:
    region_pr = regionToolset.Region(faces=p_ring.faces)
    p_ring.SectionAssignment(
        region=region_pr,
        sectionName=ring_sec_name,
        offset=0.0,
        offsetType=MIDDLE_SURFACE,
        offsetField='',
        thicknessAssignment=FROM_SECTION,
    )

region_p2 = regionToolset.Region(faces=p2.faces)
p2.SectionAssignment(
    region=region_p2,
    sectionName=long_sec_name,
    offset=0.0,
    offsetType=MIDDLE_SURFACE,
    offsetField='',
    thicknessAssignment=FROM_SECTION,
)


## 7) 装配
- 创建封头/环筋/纵筋实例。
- 以全局 Y 轴为中心，将单条纵筋阵列成 `n_long_ribs` 条。
- 将壳体，环线筋，纵向筋实例进行合并，可以规避垂直交线处的接触问题。

In [ ]:
asm = model.rootAssembly
asm.DatumCsysByDefault(CARTESIAN)

head_inst_name = 'HeadShell-1'
ring_inst_name = 'CircRingRibs-1'
rib_inst_name = 'LongitudinalRib-1'

if head_inst_name in asm.instances:
    del asm.instances[head_inst_name]
if ring_inst_name in asm.instances:
    del asm.instances[ring_inst_name]
if rib_inst_name in asm.instances:
    del asm.instances[rib_inst_name]

asm.Instance(name=head_inst_name, part=p1, dependent=ON)
if p_ring is not None:
    asm.Instance(name=ring_inst_name, part=p_ring, dependent=ON)
asm.Instance(name=rib_inst_name, part=p2, dependent=ON)

if n_long_ribs > 1:
    asm.RadialInstancePattern(
        instanceList=(rib_inst_name,),
        point=(0.0, 0.0, 0.0),
        axis=(0.0, 1.0, 0.0),
        number=n_long_ribs,
        totalAngle=360.0,
    )

